# Promotion Uplift Model: dunnhumby "The Complete Journey" (runs on Google Colab)

**Goal:** learn from real supermarket data **how much in-store displays and mailer promotions increase sales**,
per department and product category, and export those numbers for the Promotion Optimization app.

**How to run on Colab**
1. Open https://colab.research.google.com → **File → Upload notebook** → choose this file.
2. **Runtime → Run all**. No GPU needed (the default CPU runtime is fine). Takes about 5–10 minutes.
3. At the end, a file `uplift_results.zip` downloads automatically (results, model and charts).

**The data:** one year (2017) of shopping by 2,469 households at a US grocery chain (84.51° / dunnhumby),
loaded from the public `completejourney` R package on GitHub. Besides purchases, it records for every
product, store and week whether the product was **on an in-store display** and/or **in the weekly mailer**.

**The idea in simple words**
- The model (**LightGBM**, many small decision trees where each new tree fixes the previous ones' mistakes) learns:
  *product + week + recent sales + "on display?" + "in mailer?" → units sold that week*.
- **Uplift:** ask the trained model the same question with the promotion **ON** and **OFF**. The difference is the promotion effect.
- **Honest test:** the model never sees the last 8 weeks of the year while learning; we score it there, and it must beat a simple baseline.

## 1. Settings

In [ ]:
import os, sys, json, time, gc, shutil, subprocess, urllib.request
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt

try:
    import pyreadr            # reads R data files (.rds / .rda)
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyreadr'], check=True)
    import pyreadr

IN_COLAB = 'google.colab' in sys.modules
OUT_DIR  = os.environ.get('OUT_DIR', '/content/outputs' if IN_COLAB else 'outputs')
DATA_DIR = os.environ.get('CJ_DIR', '/content/data' if IN_COLAB else 'data_cj')
FIG_DIR  = os.path.join(OUT_DIR, 'figures')
for d in (OUT_DIR, DATA_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

BASE_URL = 'https://github.com/bradleyboehmke/completejourney/raw/master/data/'
FILES = ['transactions.rds', 'promotions.rds', 'products.rda']

EXCLUDE_DEPARTMENTS = ['FUEL', 'MISCELLANEOUS', 'KIOSK-GAS']  # "quantity" there is gallons / odd units
MIN_WEEKS_SOLD = 20     # keep products sold in at least 20 of the weeks before the test period
LAST_WEEK  = 52         # week 53 is only 1 day long, drop it
FIRST_ROW_WEEK = 9      # first week we predict (needs 8 weeks of history before it)
VALID_START = 39        # weeks 39-44: used to decide when to stop training
TEST_START  = 45        # weeks 45-52: never seen while learning
SEED = 42
print('Colab:', IN_COLAB, '| pandas', pd.__version__, '| lightgbm', lgb.__version__)

## 2. Download and load the data (about 37 MB)

In [ ]:
for f in FILES:
    path = os.path.join(DATA_DIR, f)
    if not os.path.exists(path):
        print('Downloading', f, '...', flush=True)
        urllib.request.urlretrieve(BASE_URL + f, path)

tx    = pyreadr.read_r(os.path.join(DATA_DIR, 'transactions.rds'))[None]
promo = pyreadr.read_r(os.path.join(DATA_DIR, 'promotions.rds'))[None]
prod  = list(pyreadr.read_r(os.path.join(DATA_DIR, 'products.rda')).values())[0]
print(f'transactions {len(tx):,} | promotions {len(promo):,} | products {len(prod):,}')
tx.head()

## 3. Clean and aggregate to **product × week**
Each household buys a product only now and then, so sales per *store* are tiny (usually 1 unit).
We add up all stores: **units of each product sold per week**.

We keep products that sold in at least 20 weeks **before the test period** (deciding with test weeks would be cheating).
The **target** is `y = log(1 + units)`, so a few best-sellers don't dominate.

In [ ]:
prod['product_id'] = prod['product_id'].astype(str)
tx['product_id'] = tx['product_id'].astype(str)
tx = tx.merge(prod[['product_id', 'department']], on='product_id', how='left')
tx = tx[~tx['department'].isin(EXCLUDE_DEPARTMENTS) & (tx['week'] <= LAST_WEEK) & (tx['quantity'] > 0)]

pw = tx.groupby(['product_id', 'week'], observed=True)['quantity'].sum().rename('units').reset_index()
weeks_sold = pw[pw['week'] < TEST_START].groupby('product_id').size()
keep = weeks_sold[weeks_sold >= MIN_WEEKS_SOLD].index
pw = pw[pw['product_id'].isin(keep)]

products = pd.Index(sorted(keep), name='product_id')
n_prod, n_weeks = len(products), LAST_WEEK
p_idx = products.get_indexer(pw['product_id'])
Y = np.zeros((n_prod, n_weeks), dtype='float32')            # log(1+units), weeks 1..52 -> columns 0..51
Y[p_idx, pw['week'].values - 1] = np.log1p(pw['units'].values)
print(f'{n_prod:,} products x {n_weeks} weeks | share of product-weeks with a sale: {(Y > 0).mean():.0%}')

## 4. Promotion features: display & mailer
For each product and week: the **share of the chain's stores** that had it on an in-store display, and the share that had it in the weekly mailer
(`0` in the data means "not on display / not in mailer"). Promotions are planned in advance, so using **this week's** promotion is fair.

In [ ]:
promo['product_id'] = promo['product_id'].astype(str)
n_promo_stores = promo['store_id'].nunique()
pr = promo[promo['product_id'].isin(products) & (promo['week'] <= LAST_WEEK)]
pr = pr.assign(disp=(pr['display_location'].astype(str) != '0'), mail=(pr['mailer_location'].astype(str) != '0'))
agg = pr.groupby(['product_id', 'week'], observed=True)[['disp', 'mail']].sum().reset_index()
D = np.zeros((n_prod, n_weeks), dtype='float32')   # share of stores with the product on display
M = np.zeros((n_prod, n_weeks), dtype='float32')   # share of stores with the product in the mailer
ai, aw = products.get_indexer(agg['product_id']), agg['week'].values - 1
D[ai, aw] = agg['disp'].values / n_promo_stores
M[ai, aw] = agg['mail'].values / n_promo_stores
del promo, pr, agg; gc.collect()
print(f'{n_promo_stores} stores | product-weeks on display somewhere: {(D > 0).mean():.0%} | in mailer: {(M > 0).mean():.0%}')

## 5. Look at the data first

In [ ]:
weekly_total = np.expm1(Y).sum(axis=0)
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.plot(range(1, n_weeks + 1), weekly_total, marker='.')
ax.axvspan(TEST_START - 0.5, LAST_WEEK + 0.5, color='tab:red', alpha=0.08, label='test weeks')
ax.set_title('Total units per week (selected products)'); ax.set_xlabel('week of 2017'); ax.legend()
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/01_weekly_units.png', dpi=120); plt.show(); plt.close('all')

dept = prod.set_index('product_id').loc[products, 'department'].values
cat  = prod.set_index('product_id').loc[products, 'product_category'].fillna('UNKNOWN').values
train_cols = slice(FIRST_ROW_WEEK - 1, VALID_START - 1)
units_tr = np.expm1(Y[:, train_cols]); on_tr = (D[:, train_cols] > 0) | (M[:, train_cols] > 0)
raw = pd.DataFrame({'department': np.repeat(dept, units_tr.shape[1]), 'units': units_tr.ravel(), 'promo': on_tr.ravel()})
g = raw.groupby(['department', 'promo'])['units'].mean().unstack()
raw_lift = (g[True] / g[False] - 1).rename('raw_promo_lift')
promo_rows = raw.groupby('department')['promo'].sum().rename('promo_weeks_in_training')
ax = (raw_lift.dropna().sort_values() * 100).plot.barh(figsize=(8, 5), title='Raw lift: avg units in promo weeks vs other weeks (%)')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/02_raw_lift.png', dpi=120); plt.show(); plt.close('all')
del raw; gc.collect()

## 6. Features: what the model is allowed to look at
**Rule against cheating (no data leakage):** sales features only use **earlier weeks**.

| Group | Features |
|---|---|
| Past sales | sales 1, 2, 3, 4 weeks ago; average of the last 4 / 8 / 13 weeks; share of recent weeks with a sale |
| Promotion | **display share** and **mailer share** this week, last week's promo, number of promo weeks in the last 8 |
| Calendar | week of year, month, holiday weeks (Super Bowl, Easter, Memorial Day, July 4, Labor Day, Thanksgiving, Christmas) |
| Product | department, product category, private label or national brand |

In [ ]:
def padded_cumsum(A):
    out = np.zeros((A.shape[0], A.shape[1] + 1), dtype='float64')
    np.cumsum(A, axis=1, out=out[:, 1:])
    return out

CY, CS = padded_cumsum(Y), padded_cumsum((Y > 0).astype('float32'))
CP = padded_cumsum(((D > 0) | (M > 0)).astype('float32'))

def past_mean(C, p, t, w):
    # average over the w weeks before week index t (fewer at the start of the year)
    lo = np.maximum(t - w, 0)
    return ((C[p, t] - C[p, lo]) / np.maximum(t - lo, 1)).astype('float32')

week_start = pd.Timestamp('2017-01-01') + pd.to_timedelta(7 * np.arange(n_weeks), unit='D')
HOLIDAYS = {'super_bowl': '2017-02-05', 'easter': '2017-04-16', 'memorial_day': '2017-05-29', 'july_4': '2017-07-04',
            'labor_day': '2017-09-04', 'thanksgiving': '2017-11-23', 'christmas': '2017-12-25'}
holiday_week = {name: np.array([(s <= pd.Timestamp(d) < s + pd.Timedelta(days=7)) for s in week_start], dtype='int8')
                for name, d in HOLIDAYS.items()}

pinfo = prod.set_index('product_id').loc[products]
dept_codes = pd.Categorical(pinfo['department']); cat_codes = pd.Categorical(pinfo['product_category'].fillna('UNKNOWN'))
DEPT_NAMES, CAT_NAMES = list(dept_codes.categories), list(cat_codes.categories)
dept_code, cat_code = dept_codes.codes, cat_codes.codes
private = (pinfo['brand'].astype(str) == 'Private').astype('int8').values
CAT_COLS = ['department_code', 'category_code']

def build_features(p, t, disp=None, mail=None):
    f = pd.DataFrame({f'lag_{k}': Y[p, t - k] for k in (1, 2, 3, 4)})
    for w in (4, 8, 13):
        f[f'mean_{w}w'] = past_mean(CY, p, t, w)
    f['sold_share_8w'] = past_mean(CS, p, t, 8)
    f['display_share'] = D[p, t] if disp is None else disp
    f['mailer_share']  = M[p, t] if mail is None else mail
    f['promo_last_week'] = ((D[p, t - 1] > 0) | (M[p, t - 1] > 0)).astype('int8')
    f['promo_weeks_8w'] = past_mean(CP, p, t, 8)
    f['week'] = t + 1
    f['month'] = week_start[t].month.values
    for name, arr in holiday_week.items():
        f[name] = arr[t]
    f['department_code'] = dept_code[p]
    f['category_code'] = cat_code[p]
    f['private_label'] = private[p]
    return f

# one row per (product, week) from week 9 to 52
pp, tt = np.meshgrid(np.arange(n_prod), np.arange(FIRST_ROW_WEEK - 1, n_weeks), indexing='ij')
pp, tt = pp.ravel(), tt.ravel()
week_no = tt + 1
split = {'train': week_no < VALID_START, 'valid': (week_no >= VALID_START) & (week_no < TEST_START), 'test': week_no >= TEST_START}
X = {k: build_features(pp[m], tt[m]) for k, m in split.items()}
y = {k: Y[pp[m], tt[m]] for k, m in split.items()}
FEATURES = list(X['train'].columns)
print({k: len(v) for k, v in X.items()}, '|', len(FEATURES), 'features')

## 7. Baseline: the "simple guess" to beat
Guess = **average of the last 4 weeks**.
Scores (on the test weeks 45–52, never seen in training):
- **RMSLE**: error on the log scale. **Lower is better.**
- **MAE**: average error in real units.
- **WAPE**: total absolute error ÷ total units (as a %).

In [ ]:
def score(y_true, pred):
    a, b = np.expm1(y_true), np.expm1(pred)
    return {'rmsle': float(np.sqrt(np.mean((pred - y_true) ** 2))),
            'mae_units': float(np.mean(np.abs(b - a))),
            'wape': float(np.abs(b - a).sum() / a.sum())}

baseline = X['test']['mean_4w'].values
scores = {'baseline_4w_mean': score(y['test'], baseline)}
print(scores)

## 8. Train LightGBM
Trees are added one by one; every 100 trees we check the error on the **validation** weeks (39–44).
If it hasn't improved for 200 rounds, training stops (**early stopping**), so the model doesn't memorise the training weeks (**overfitting**).

In [ ]:
params = dict(objective='regression', metric='rmse', learning_rate=0.03, num_leaves=63, min_data_in_leaf=100,
              feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, seed=SEED, verbose=-1)
dtrain = lgb.Dataset(X['train'], y['train'], categorical_feature=CAT_COLS)
dvalid = lgb.Dataset(X['valid'], y['valid'], categorical_feature=CAT_COLS, reference=dtrain)
t0 = time.time()
model = lgb.train(params, dtrain, num_boost_round=5000, valid_sets=[dvalid], valid_names=['valid'],
                  callbacks=[lgb.early_stopping(200), lgb.log_evaluation(100)])
print(f'Best number of trees: {model.best_iteration} | {time.time() - t0:.0f}s')

pred = model.predict(X['test'], num_iteration=model.best_iteration).clip(0)
scores['lightgbm'] = score(y['test'], pred)
is_promo = (X['test']['display_share'].values > 0) | (X['test']['mailer_share'].values > 0)
for name, m in [('promo_weeks', is_promo), ('normal_weeks', ~is_promo)]:
    scores['baseline_4w_mean'][f'rmsle_{name}'] = score(y['test'][m], baseline[m])['rmsle']
    scores['lightgbm'][f'rmsle_{name}'] = score(y['test'][m], pred[m])['rmsle']
print(pd.DataFrame(scores).T.round(4))
improvement = 1 - scores['lightgbm']['rmsle'] / scores['baseline_4w_mean']['rmsle']
print(f'\nLightGBM error (RMSLE) is {improvement:.1%} lower than the baseline on weeks {TEST_START}-{LAST_WEEK}.')
model.save_model(f'{OUT_DIR}/lgb_model.txt', num_iteration=model.best_iteration)

## 8b. How accurate is it? Three levels
The numbers above are for **one product in one week**: very small numbers (often 0–3 units), so they're noisy.
Planners usually care about **whole departments**, so we also add up predictions and real sales:
- **product-week**: each product, each week
- **department-week**: all products of a department, each week
- **total-week**: the whole store, each week

**Forecast accuracy = 1 − WAPE** = 1 − (total absolute error ÷ total real units). 100% is perfect; higher is better.

**Bias correction:** the model learns `log(1 + units)`. Turning that back into units gives predictions that are
**too low on average** (a known effect of log models), and when we add up a whole department those small misses add up.
We measure how much too low on the **validation weeks** (never the test weeks) and multiply by that one factor.
We do the same for the baseline, so the comparison stays fair. **Uplift numbers don't change**: they're ratios (with ÷ without), so the factor cancels out.

In [ ]:
pred_valid = model.predict(X['valid'], num_iteration=model.best_iteration).clip(0)
calib = {'model': float(np.expm1(y['valid']).sum() / np.expm1(pred_valid).sum()),
         'baseline': float(np.expm1(y['valid']).sum() / np.expm1(X['valid']['mean_4w'].values).sum())}
print('Correction factors (from validation weeks):', {k: round(v, 3) for k, v in calib.items()})

test_p, test_w = pp[split['test']], tt[split['test']] + 1
acc = pd.DataFrame({'department': np.array(DEPT_NAMES)[dept_code[test_p]], 'week': test_w,
                    'actual': np.expm1(y['test']),
                    'model': np.expm1(pred) * calib['model'],
                    'baseline': np.expm1(baseline) * calib['baseline']})

def level_scores(df):
    out = {}
    for m in ['model', 'baseline']:
        err = np.abs(df[m] - df['actual'])
        out[m] = {'accuracy': float(1 - err.sum() / df['actual'].sum()),
                  'mae_units': float(err.mean()),
                  'rmsle': float(np.sqrt(np.mean((np.log1p(df[m]) - np.log1p(df['actual'])) ** 2)))}
    return out

dept_week = acc.groupby(['department', 'week'])[['actual', 'model', 'baseline']].sum()
accuracy_levels = {
    'product_week':    level_scores(acc),
    'department_week': level_scores(dept_week),
    'total_week':      level_scores(acc.groupby('week')[['actual', 'model', 'baseline']].sum()),
}
lv = pd.DataFrame({lvl: {'model': s['model']['accuracy'], 'baseline': s['baseline']['accuracy']}
                   for lvl, s in accuracy_levels.items()}).T
print((lv * 100).round(1).rename(columns=lambda c: c + ' accuracy %'))

acc_dept = dept_week.groupby('department').apply(lambda d: pd.Series({
    'test_units': d['actual'].sum(),
    'accuracy_model': 1 - np.abs(d['model'] - d['actual']).sum() / d['actual'].sum(),
    'accuracy_baseline': 1 - np.abs(d['baseline'] - d['actual']).sum() / d['actual'].sum()}))
acc_dept = acc_dept.sort_values('test_units', ascending=False).round(4)
acc_dept.to_csv(f'{OUT_DIR}/accuracy_by_department.csv')
print(acc_dept.to_string())

ax = (lv * 100).plot.bar(figsize=(7, 4), rot=0, color=['#4F46E5', '#A5B4FC'],
                         title='Forecast accuracy (1 - WAPE) on test weeks, %')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/06_accuracy_levels.png', dpi=120); plt.show(); plt.close('all')

## 9. What did the model rely on?

In [ ]:
imp = pd.Series(model.feature_importance('gain'), index=FEATURES).sort_values()
(imp / imp.sum() * 100).tail(20).plot.barh(figsize=(8, 7), title='Feature importance (% of total gain)')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/03_feature_importance.png', dpi=120); plt.show(); plt.close('all')

plt.figure(figsize=(5, 5))
plt.scatter(y['test'], pred, s=3, alpha=0.2)
lim = float(max(y['test'].max(), pred.max()))
plt.plot([0, lim], [0, lim], 'r--', lw=1)
plt.xlabel('actual log(1+units)'); plt.ylabel('predicted'); plt.title('Test weeks: predicted vs actual')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/04_pred_vs_actual.png', dpi=120); plt.show(); plt.close('all')

## 10. Promotion uplift
For every test row we ask the model 4 times, changing **only** the promotion:
- **OFF**: no display, no mailer
- **Display only**, **Mailer only**, **Both**: at the *typical* store coverage for that department (the median seen in training)

`uplift = total predicted units with the promotion ÷ total with OFF − 1`

**Ranges:** we resample the products of each department 300 times and recompute the uplift.
The `_low` / `_high` columns hold the 90% range: a wide range means the number depends a lot on which products are included.

**Honest limitation:** the retailer didn't choose promotions at random, so this is a *model estimate* of the effect, not proven cause and effect.
Groups with few promotion weeks in training are marked `low_confidence`.

In [ ]:
pt, tt_test = pp[split['test']], tt[split['test']]
tr_p, tr_t = pp[split['train']], tt[split['train']]
d_tr, m_tr = D[tr_p, tr_t], M[tr_p, tr_t]
typ = pd.DataFrame({'dept': dept_code[tr_p], 'd': np.where(d_tr > 0, d_tr, np.nan), 'm': np.where(m_tr > 0, m_tr, np.nan)})
typ = typ.groupby('dept')[['d', 'm']].median()
glob_d, glob_m = np.nanmedian(np.where(d_tr > 0, d_tr, np.nan)), np.nanmedian(np.where(m_tr > 0, m_tr, np.nan))
typ_d = typ['d'].reindex(dept_code[pt]).fillna(glob_d).values.astype('float32')
typ_m = typ['m'].reindex(dept_code[pt]).fillna(glob_m).values.astype('float32')

zero = np.zeros(len(pt), dtype='float32')
scenarios = {'off': (zero, zero), 'display': (typ_d, zero), 'mailer': (zero, typ_m), 'both': (typ_d, typ_m)}
units = {k: np.expm1(model.predict(build_features(pt, tt_test, d, m), num_iteration=model.best_iteration).clip(0))
         for k, (d, m) in scenarios.items()}

def uplift_table(group_names):
    u = pd.DataFrame({k: v for k, v in units.items()}); u['group'] = group_names
    t = u.groupby('group').sum()
    out = pd.DataFrame({f'uplift_{k}': t[k] / t['off'] - 1 for k in ['display', 'mailer', 'both']})
    out['products'] = pd.Series(group_names).groupby(group_names).size() // (LAST_WEEK - TEST_START + 1)
    return out

# Ranges: resample the products of each group 300 times ("what if we had a different set of products?")
per_prod = {k: np.bincount(pt, weights=v, minlength=n_prod) for k, v in units.items()}

def boot_ranges(codes, names, B=300):
    rng, rows = np.random.default_rng(SEED), []
    for g in np.unique(codes):
        members = np.where(codes == g)[0]
        idx = members[rng.integers(0, len(members), size=(B, len(members)))]
        off = per_prod['off'][idx].sum(axis=1)
        row = {'group': names[g]}
        for k in ['display', 'mailer', 'both']:
            r = per_prod[k][idx].sum(axis=1) / np.maximum(off, 1e-9) - 1
            row[f'uplift_{k}_low'], row[f'uplift_{k}_high'] = np.percentile(r, [5, 95])
        rows.append(row)
    return pd.DataFrame(rows).set_index('group')

dept_up = uplift_table(np.array(DEPT_NAMES)[dept_code[pt]]).join(boot_ranges(dept_code, DEPT_NAMES)).join(raw_lift).join(promo_rows)
dept_up['promo_weeks_in_training'] = dept_up['promo_weeks_in_training'].fillna(0).astype(int)
dept_up['low_confidence'] = dept_up['promo_weeks_in_training'] < 200
dept_up = dept_up.sort_values('uplift_both', ascending=False).round(4)
dept_up.index.name = 'department'
dept_up.to_csv(f'{OUT_DIR}/department_uplift.csv')

cat_up = uplift_table(np.array(CAT_NAMES)[cat_code[pt]]).join(boot_ranges(cat_code, CAT_NAMES))
cat_up['low_confidence'] = cat_up['products'] < 10
cat_up.index.name = 'product_category'
cat_up.sort_values('uplift_both', ascending=False).round(4).to_csv(f'{OUT_DIR}/category_uplift.csv')

overall = {k: float(units[k].sum() / units['off'].sum() - 1) for k in ['display', 'mailer', 'both']}
print('Overall uplift:', {k: f'{v:.1%}' for k, v in overall.items()})
print(dept_up.to_string())

dept_up[['uplift_display', 'uplift_mailer', 'uplift_both']].mul(100).sort_values('uplift_both').plot.barh(
    figsize=(9, 6), title='Model-estimated promotion uplift by department (%)')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/05_department_uplift.png', dpi=120); plt.show(); plt.close('all')

## 11. Save and download the results
`product_catalog.csv` lists the products the model learned from. The Streamlit app uses it as its catalog (`items.csv`),
with columns `item_nbr` (product id), `family` (department), `class` (product category), `perishable` and `brand`.

In [ ]:
PERISHABLE_DEPARTMENTS = {'PRODUCE', 'MEAT', 'MEAT-PCKGD', 'DELI', 'PASTRY', 'SEAFOOD', 'SEAFOOD-PCKGD',
                          'SALAD BAR', 'FLORAL', 'CHEF SHOPPE', 'RESTAURANT'}
catalog = pinfo.reset_index()[['product_id', 'department', 'product_category', 'brand']]
catalog.columns = ['item_nbr', 'family', 'class', 'brand']
catalog['class'] = catalog['class'].fillna('UNKNOWN')
catalog['brand'] = catalog['brand'].astype(str)
catalog.insert(3, 'perishable', catalog['family'].isin(PERISHABLE_DEPARTMENTS).astype(int))
catalog.to_csv(f'{OUT_DIR}/product_catalog.csv', index=False)
print(f'product_catalog.csv: {len(catalog):,} products in {catalog.family.nunique()} departments')

metrics = {
    'data': {'source': 'dunnhumby / 84.51 The Complete Journey (completejourney R package)', 'products': int(n_prod),
             'weeks': [1, LAST_WEEK], 'train_weeks': [FIRST_ROW_WEEK, VALID_START - 1], 'valid_weeks': [VALID_START, TEST_START - 1],
             'test_weeks': [TEST_START, LAST_WEEK], 'rows': {k: int(len(v)) for k, v in X.items()},
             'promo_stores': int(n_promo_stores), 'features': FEATURES},
    'model': {'type': 'LightGBM', 'params': params, 'best_iteration': int(model.best_iteration)},
    'scores_test': scores,
    'accuracy_levels': accuracy_levels,
    'bias_correction_factors': {k: round(v, 4) for k, v in calib.items()},
    'improvement_vs_baseline_rmsle': round(improvement, 4),
    'overall_uplift': {k: round(v, 4) for k, v in overall.items()},
}
with open(f'{OUT_DIR}/metrics.json', 'w') as fh:
    json.dump(metrics, fh, indent=2)
print(json.dumps({k: metrics[k] for k in ['accuracy_levels', 'improvement_vs_baseline_rmsle', 'overall_uplift']}, indent=2))

zip_path = shutil.make_archive(os.path.join(os.path.dirname(OUT_DIR) or '.', 'uplift_results'), 'zip', OUT_DIR)
print('Saved', zip_path)
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)